Import libraries

In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import TensorDataset,DataLoader 

Data Loading

In [5]:
df=pd.read_csv(r"C:\Datasets\IMDB Dataset.csv")

Data Inspection

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df.shape

(50000, 2)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [10]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
49995    False
49996    False
49997    False
49998    False
49999    False
Length: 50000, dtype: bool

In [11]:
df.drop_duplicates(inplace=True)

Pre Processing

1:Convert To Lowercase

In [12]:
df["review"]=df["review"].str.lower()

2:Removing URLs

In [13]:
import re
def remove_urls(text):
    text=re.sub(r"http\S+","",text) #(patteren,replace,string) e.g --https://www.google.com
    return text
df["review"]=df["review"].apply(remove_urls)

3:Removing functuation

In [14]:
def remove_puctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text) 
    return text
df["review"]=df["review"].apply(remove_puctuations)  

4:Removing HTML Tags

In [15]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text) 
    return text
df["review"]=df["review"].apply(remove_html)

5:Remove stopwords

In [16]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [17]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text

df["review"]=df["review"].apply(remove_stopwords)

6:Steaming

In [18]:
# running->run
# played ->play
from nltk.stem import PorterStemmer

In [19]:
def steaming(text):
    ps=PorterStemmer()
    steammed_words=[]
    
    tokens=word_tokenize(text)
    for token in tokens:
        steammed_token=ps.stem(token)
        steammed_words.append(steammed_token)
    return " ".join(steammed_words)
df["review"]=df["review"].apply(steaming) 

7:Encoding For Target Values

In [20]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]

8:Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

Dataset & Data Loader

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [24]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [25]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [26]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

Build RNN

In [29]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, X):
        # Initial hidden state
        h0 = torch.zeros(
            self.num_layers,
            X.size(0),
            self.hidden_size
        )

        # RNN
        out, _ = self.rnn(X, h0)

        # Last time step
        out = self.fc(out[:, -1, :])

        return out

In [30]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

Training RNN

In [34]:
epochs=10
for epoch in range(epochs):
    model.train()
    
    for xb,yb in train_loader:
        optimizer.zero_grad()
        
        xb=xb.unsqueeze(1) #add singleton direction
        outputs=model(xb) #(batch_size,1)
        
        outputs=torch.sigmoid(outputs.squeeze()) #(batch_size,)=>probability
        
        loss=criterion(outputs,yb) #compute loss
        loss.backward() #backpropagation
         
        optimizer.step() #update weights
        
    print(f"epoch={epoch+1}/{epochs} and loss={loss.item()}")

epoch=1/10 and loss=0.4224925637245178
epoch=2/10 and loss=0.3923269510269165
epoch=3/10 and loss=0.25183504819869995
epoch=4/10 and loss=0.36016973853111267
epoch=5/10 and loss=0.3535732328891754
epoch=6/10 and loss=0.33771297335624695
epoch=7/10 and loss=0.22668132185935974
epoch=8/10 and loss=0.3110381066799164
epoch=9/10 and loss=0.2079869508743286
epoch=10/10 and loss=0.17320778965950012


Evaluation

In [35]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.70132096400121


In [36]:
import pickle

# 1. Save the trained model weights + architecture params
torch.save({
    "model_state_dict": model.state_dict(),
    "input_size": input_size,
    "hidden_size": model.hidden_size,
    "num_layers": model.num_layers
}, "rnn_sentiment_model.pth")

# 2. Save the TF-IDF vectorizer (needed to convert new text into the same feature space)
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)

# 3. Save the label encoder (needed to convert predictions back to "positive"/"negative")
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)